模型回归 （用合作对象对专利费用比，新产品质量，人工智能专利数回归）

In [3]:
import pandas as pd
import networkx as nx

# 读取数据
df = pd.read_csv(r'/home/mw/input/data_total74037403/行业角度数据集/合作对象占比.csv')

years = sorted(df['年份'].unique())
enterprise_nodes = ['大型企业', '中型企业', '小型企业', '微型企业']
partners = list(df.columns[2:])

centrality_list = []

for year in years:
    df_year = df[df['年份'] == year]
    # 构建网络
    G = nx.Graph()
    G.add_nodes_from(enterprise_nodes)
    G.add_nodes_from(partners)
    for p in partners:
        for ent in enterprise_nodes:
            value = df_year[df_year['企业规模'] == ent][p].values
            if len(value) > 0:
                G.add_edge(ent, p, weight=value[0])
    # 计算中心性
    degree_centrality = nx.degree_centrality(G)
    closeness_centrality = nx.closeness_centrality(
        G, distance=lambda u, v, d: 1/d['weight'] if d['weight'] > 0 else float('inf')
    )
    for ent in enterprise_nodes:
        centrality_list.append({
            '年份': year,
            '企业规模': ent,
            '度中心性': degree_centrality.get(ent, 0),
            '接近中心性': closeness_centrality.get(ent, 0)
        })

centrality_df = pd.DataFrame(centrality_list)

In [4]:
import pandas as pd

# 读取数据
df_coop = pd.read_csv(r'/home/mw/input/data_total74037403/行业角度数据集/合作对象占比.csv')
df_inno = pd.read_csv(r'/home/mw/input/data_total74037403/行业角度数据集/全国企业创新结果.csv')

# 合并所有年份
result = pd.merge(df_coop, df_inno, on=['年份', '企业规模'], how='left')
#result = pd.merge(result, closeness_df[['年份', '企业规模', '接近中心性']], on=['年份', '企业规模'], how='left')
result = pd.merge(result, centrality_df[['年份', '企业规模', '度中心性']], on=['年份', '企业规模'], how='left')

result['技术合作对象'] = (
    result['集团内其他企业'] +
    result['高等学校'] +
    result['研究机构'] +
    result['政府部门或行业协会']
)
result['应用合作对象'] = (
    result['供应商'] +
    result['客户'] +
    result['竞争对手或同行业企业'] +
    result['咨询顾问，市场分析及中介机构']
)

result = result.drop(['集团内其他企业', '高等学校', '研究机构', '政府部门或行业协会'], axis=1)
result = result.drop(['供应商','客户','竞争对手或同行业企业','咨询顾问，市场分析及中介机构','其他'], axis=1)


print(result)

      年份  企业规模     企业数  开展产学研合作的企业数  在创新合作企业中产学研合作企业占比  创新费用支出合计(亿元)   专利总计  \
0   2016  大型企业   21166         5445               57.1        9992.6  11195   
1   2016  中型企业  144807        12116               40.4        3737.3   4408   
2   2016  小型企业  491556        28718               37.1        3665.0    101   
3   2016  微型企业   68610          770               23.5          84.3      0   
4   2017  大型企业   21376         5735               56.5       10695.6   9805   
5   2017  中型企业  140701        12699               40.0        4158.9   5849   
6   2017  小型企业  503115        30988               36.5        4233.1    143   
7   2017  微型企业   83653         1040               25.4          58.4      0   
8   2018  大型企业   21213         5774               55.5       11653.6  12531   
9   2018  中型企业  134179        12226               38.6        4345.6   6726   
10  2018  小型企业  465953        29966               34.6        4195.9    160   
11  2018  微型企业  133241         4499               33

In [5]:
import pandas as pd
from linearmodels.panel import PanelOLS

# 读取数据
df = result.copy()

# 设置多级索引（企业规模和年份）
df = df.set_index(['企业规模', '年份'])

# 构建模型
exog_vars = ['创新费用支出合计(亿元)', '开展产学研合作的企业数','技术合作对象']
exog = df[exog_vars]
endog = df['新产品销售收入(亿元)']

# 固定效应回归（控制企业规模和年份的固定效应）
mod = PanelOLS(endog, exog, entity_effects=True, time_effects=True)
res = mod.fit()
print(res.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:            新产品销售收入(亿元)   R-squared:                        0.9272
Estimator:                   PanelOLS   R-squared (Between):              0.8147
No. Observations:                  32   R-squared (Within):               0.9489
Date:                Fri, May 30 2025   R-squared (Overall):              0.8203
Time:                        01:45:51   Log-likelihood                   -309.24
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      76.380
Entities:                           4   P-value                           0.0000
Avg Obs:                       8.0000   Distribution:                    F(3,18)
Min Obs:                       8.0000                                           
Max Obs:                       8.0000   F-statistic (robust):             76.380
                            

In [6]:
import pandas as pd
import matplotlib.pyplot as plt

# 提取主要回归系数表
summary_df = res.summary.tables[1]
summary_df = pd.DataFrame(summary_df.data[1:], columns=summary_df.data[0])

# 提取统计量表
stat_df = res.summary.tables[0]
stat_df = pd.DataFrame(stat_df.data[1:], columns=stat_df.data[0])

# 绘制回归系数表为图片
fig1, ax1 = plt.subplots(figsize=(8, 2 + 0.5*len(summary_df)))
ax1.axis('off')
tbl1 = ax1.table(
    cellText=summary_df.values,
    colLabels=summary_df.columns,
    loc='center',
    cellLoc='center'
)
tbl1.auto_set_font_size(False)
tbl1.set_fontsize(12)
tbl1.scale(1.2, 1.2)
# 延长第一列格子宽度
for row in range(len(summary_df)+1):  
    tbl1[(row, 0)].set_width(0.35)
plt.tight_layout()
plt.show()

# 绘制统计量表为图片
fig2, ax2 = plt.subplots(figsize=(5, 2 + 0.5*len(stat_df)))
ax2.axis('off')
tbl2 = ax2.table(
    cellText=stat_df.values,
    colLabels=stat_df.columns,
    loc='center',
    cellLoc='center'
)
tbl2.auto_set_font_size(False)
tbl2.set_fontsize(12)
tbl2.scale(1.2, 1.2)
# 延长所有格子宽度
for row in range(len(stat_df)+1):  
    for col in range(len(stat_df.columns)):
        tbl2[(row, col)].set_width(0.6)
plt.tight_layout()
plt.show()

<Figure size 800x350 with 1 Axes>

<Figure size 500x1000 with 1 Axes>

In [7]:
import pandas as pd
from linearmodels.panel import PanelOLS

# 读取数据
df = result.copy()

# 设置多级索引（企业规模和年份）
df = df.set_index(['企业规模', '年份'])

# 构建模型
exog_vars = ['创新费用支出合计(亿元)', '开展产学研合作的企业数','技术合作对象']
exog = df[exog_vars]
endog = df['新产品销售收入占主营业务收入的比重']

# 固定效应回归（控制企业规模和年份的固定效应）
mod = PanelOLS(endog, exog, entity_effects=True, time_effects=True)
res = mod.fit()
print(res.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:      新产品销售收入占主营业务收入的比重   R-squared:                        0.7027
Estimator:                   PanelOLS   R-squared (Between):              0.8289
No. Observations:                  32   R-squared (Within):               0.5041
Date:                Fri, May 30 2025   R-squared (Overall):              0.8254
Time:                        01:45:57   Log-likelihood                   -29.499
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      14.184
Entities:                           4   P-value                           0.0001
Avg Obs:                       8.0000   Distribution:                    F(3,18)
Min Obs:                       8.0000                                           
Max Obs:                       8.0000   F-statistic (robust):             14.184
                            

In [8]:
import pandas as pd
import matplotlib.pyplot as plt

# 提取主要回归系数表
summary_df = res.summary.tables[1]
summary_df = pd.DataFrame(summary_df.data[1:], columns=summary_df.data[0])

# 提取统计量表
stat_df = res.summary.tables[0]
stat_df = pd.DataFrame(stat_df.data[1:], columns=stat_df.data[0])

# 绘制回归系数表为图片
fig1, ax1 = plt.subplots(figsize=(8, 2 + 0.5*len(summary_df)))
ax1.axis('off')
tbl1 = ax1.table(
    cellText=summary_df.values,
    colLabels=summary_df.columns,
    loc='center',
    cellLoc='center'
)
tbl1.auto_set_font_size(False)
tbl1.set_fontsize(12)
tbl1.scale(1.2, 1.2)
# 延长第一列格子宽度
for row in range(len(summary_df)+1):  
    tbl1[(row, 0)].set_width(0.35)
plt.tight_layout()
plt.show()

# 绘制统计量表为图片
fig2, ax2 = plt.subplots(figsize=(5, 2 + 0.5*len(stat_df)))
ax2.axis('off')
tbl2 = ax2.table(
    cellText=stat_df.values,
    colLabels=stat_df.columns,
    loc='center',
    cellLoc='center'
)
tbl2.auto_set_font_size(False)
tbl2.set_fontsize(12)
tbl2.scale(1.2, 1.2)
# 延长所有格子宽度
for row in range(len(stat_df)+1):  
    for col in range(len(stat_df.columns)):
        tbl2[(row, col)].set_width(0.75)
plt.tight_layout()
plt.show()

<Figure size 576x252 with 1 Axes>

<Figure size 360x720 with 1 Axes>